In [1]:
import carla
import numpy as np
import cv2
import time


In [2]:
CARLA_HOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 10.0

IMAGE_WIDTH = 1280
IMAGE_HEIGHT = 720
CAMERA_POS_Z = 3
CAMERA_POS_X = 0

In [3]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT)
    world = client.get_world()

    if not world:
        raise RuntimeError(
            "Failed to return a valid world\n"
        )

    return client, world


In [4]:
def define_spawn_point(world: carla.World) -> carla.Transform:

    spawn_points = world.get_map().get_spawn_points()

    spawn_point = spawn_points[0]

    return spawn_point

In [5]:
def spawn_ego_vehicle(world: carla.World, spawn_point: carla.Transform) -> carla.Vehicle:

    vehicle_bp = world.get_blueprint_library().filter("vehicle.*")

    vehicle =  world.spawn_actor(
        vehicle_bp[0],
        spawn_point
    )

    return vehicle

In [6]:
def ego_vehicle_autopilot(vehicle: carla.Vehicle, autopilot_status: bool):

     vehicle.set_autopilot(
            autopilot_status,
        )
        

In [33]:
def spawn_depth_camera(world: carla.World, vehicle: carla.Vehicle) -> carla.Sensor:

    depth_sensor_bp = world.get_blueprint_library().filter("sensor.camera.depth")

    if not depth_sensor_bp:
        raise RuntimeError(
            "Carla was not able to return a valid depth sensor blueprint\n"
        )


    camera_init_trans = carla.Transform(carla.Location(z = CAMERA_POS_Z, x = CAMERA_POS_X))
    depth_sensor = world.spawn_actor(depth_sensor_bp[0], camera_init_trans, attach_to=vehicle)
    
    return depth_sensor

In [38]:
def depth_callback(image, data_dict):

    image.convert(carla.ColorConverter.Depth)
    data_dict['depth_image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

In [43]:
def main():

    client, world = connect_to_carla()

    spawn_point = define_spawn_point(world)

    vehicles = world.get_actors().filter("vehicle.*")
    sensors = world.get_actors().filter("sensor.*")

    print("Destroying active actors\n")

    for vehicle in vehicles:
        print(f"Vehicle: {vehicle.id}")
        vehicle.destroy()

    for sensor in sensors:
        print(f"Sensor: {sensor.id}")
        sensor.destroy()

    vehicle = spawn_ego_vehicle(
        world,
        spawn_point
    )

    ego_vehicle_autopilot(
        vehicle,
        True
    )

    depth_camera = spawn_depth_camera(
        world,
        vehicle
    )

    print(depth_camera)

    data_dict = {
        "depth_image": None
    }

    depth_camera.listen(
        lambda image: depth_callback(
            image,
            data_dict
        )
    )

    print("\n=== LIVE DEPTH CAMERA ===\n")
    print("Press Q to exit.\n")

    while True:

        depth_image = data_dict.get("depth_image")
        print(f"Depth image: {depth_image}")
        if depth_image is not None:
            cv2.imshow("Depth camera", depth_image)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    depth_camera.stop()
    depth_camera.destroy()

    vehicle.destroy()

    cv2.destroyAllWindows()

In [44]:
if __name__ == "__main__":
    main()

Destroying active actors

Actor(id=39, type=sensor.camera.depth)

=== LIVE DEPTH CAMERA ===

Press Q to exit.

Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: None
Depth image: [[[ 10  10  10 255]
  [ 10  10  10 255]
  [  1   1   1 255]
  ...
  [255 255 255 255]
  [255 255 255 255]
  [255 255 255 255]]

 [[ 10  10  10 255]
  [ 10  10  10 255]
  [ 10  10  10 255]
  ...
  [255 255 255 255]
  [255 255 255 255]
  [255 255 255 255]]

 [[ 10  10  10 255]
  [ 10  10  10 255]
  [ 10  10  10 255]
  ...
  [255 255 255